In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import roc_curve, auc
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.feature_selection import RFECV
from sklearn.metrics import (accuracy_score, recall_score, precision_score, f1_score,
                             confusion_matrix, brier_score_loss, roc_curve, auc,
                             average_precision_score)

Ручной перебор

In [ ]:
#загружаем данные
predict_df = pd.read_excel('датафрейм с предикторами 19_04_26 с Killip.xlsx')
predict_df['КШ развился в реанимации'].value_counts()

КШ развился в реанимации
0    5682
1     200
Name: count, dtype: int64

In [ ]:
from xgboost import XGBClassifier


predict_df = pd.read_excel('/content/датафрейм с предикторами 19_04_26 с Killip.xlsx')

def compute_confidence_interval(data, confidence=0.95):
    mean = np.mean(data)
    sem = stats.sem(data)
    interval = sem * stats.t.ppf((1 + confidence) / 2, len(data) - 1)
    return mean, mean - interval, mean + interval

def format_confidence_interval(mean, lower, upper):
    return f"{mean:.4f} [{lower:.4f}; {upper:.4f}]"

#отобрала по логистической модели
#исключить схожие признаки чтобы не было мультиколлинеарности
features = [
    'GRACE(Рассчет)',
    'Средний объем тромбоцита (MPV)',
    'SpO2',
    'Нейтрофилы (относительное значение)',
    'ФВ ЛЖ',
    'Apache II',
    'ХОБЛ',
    'Лимфоциты (абсолютное значение)',
    'Глюкоза в мг/дл',
    'Базофилы (абсолютное значение)',
    'Лейкоциты(a)',
    'Диастолического АД(b)',
    'PLR (тромбоциты/лимфоциты абс) (61-239)',
    'TIMI категория'
    ,'APACHE2 3дня категория_Низкий риск (0-10)'
]


'''убрать признаки входящие в GRACE: Age, ЧСС (b), Систолическое АД(b), Креатинин, Killip
также не использовать : 'Норадреналин', 'Отек легких', адреналин допмин добутамин и др'''


predict_df_clean = predict_df.dropna(subset=features)  #удаляем строки с NaN значениями

#подготовка данных
X = predict_df_clean[features]
y = predict_df_clean['КШ развился в реанимации']



results_data = []  #список для хранения результатов по каждому признаку
n_repeats = 100  #количество повторений разбиения на 80/20
n_splits = 5  #количество фолдов в кросс-валидации
all_roc_auc_cv = [] #хранит все roc_auc на кросс-валидации
all_roc_auc_test = [] #хранит все roc_auc на тестовой выборке
all_pr_auc_test = [] # Это и есть RAUC / PR-AUC
all_sensitivity_test = []
all_specificity_test = []
all_f1_test = []
all_ppv_test = []
all_npv_test = []
all_brier_test = []


models = []

for i in range(n_repeats):
    np.random.seed(i + 42)
    #разделение на обучающую и тестовую выборки (80/20)
    x_train, x_test, y_train, y_test = train_test_split(X, y, train_size=0.8, stratify=y, random_state=i + 42)


    model = XGBClassifier(learning_rate=0.0001)

    #кросс-валидация на 80% выборке (x_train)
    cv_results = cross_validate(model, x_train, y_train, cv=StratifiedKFold(n_splits=n_splits),
                                    scoring='roc_auc', return_estimator=True) #  Важно: return_estimator=True
    #сохраняем roc auc с каждой итерации кросс-валидации
    all_roc_auc_cv.extend(cv_results['test_score'])

    #обучение модели на всей обучающей выборке (x_train)
    model.fit(x_train, y_train)






    # 1. Получаем вероятности для ТРЕНИРОВОЧНОЙ выборки, чтобы найти порог
    y_train_prob = model.predict_proba(x_train)[:, 1]

    # Находим оптимальный порог через индекс Юдена на трейне
    fpr_train, tpr_train, thresholds_train = roc_curve(y_train, y_train_prob)
    # Индекс Юдена: Sensitivity + Specificity - 1  =>  tpr - fpr
    optimal_idx = np.argmax(tpr_train - fpr_train)
    best_threshold = thresholds_train[optimal_idx]

    # 2. Получаем вероятности для ТЕСТОВОЙ выборки
    y_test_prob = model.predict_proba(x_test)[:, 1]

    # ПРИМЕНЯЕМ ПОДОБРАННЫЙ ПОРОГ вместо стандартного 0.5
    y_test_pred = (y_test_prob >= best_threshold).astype(int)



    #ROC-AUC
    fpr, tpr, _ = roc_curve(y_test, y_test_prob)
    all_roc_auc_test.append(auc(fpr, tpr))

    #PR-AUC (часто называют RAUC или Average Precision)
    all_pr_auc_test.append(average_precision_score(y_test, y_test_prob))

    #Sensitivity
    all_sensitivity_test.append(recall_score(y_test, y_test_pred))

    #Специфичность и NPV (через матрицу ошибок)
    tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred, labels=[0, 1]).ravel()

    all_specificity_test.append(tn / (tn + fp) if (tn + fp) > 0 else 0)
    all_npv_test.append(tn / (tn + fn) if (tn + fn) > 0 else 0)

    #PPV (Precision)
    all_ppv_test.append(precision_score(y_test, y_test_pred, zero_division=0))

    #F1-score
    all_f1_test.append(f1_score(y_test, y_test_pred, zero_division=0))

    #Brier Score
    all_brier_test.append(brier_score_loss(y_test, y_test_prob))

    models.append(model)  #сохраняем модель

#находим индекс лучшей модели по тестовому ROC-AUC
best_idx = all_roc_auc_test.index(max(all_roc_auc_test))
best_model = models[best_idx]

print(f"Лучшая модель — итерация {best_idx + 1} с ROC-AUC на тесте = {all_roc_auc_test[best_idx]:.4f}")

#расчет доверительных интервалов
roc_auc_cv_mean, roc_auc_cv_lower, roc_auc_cv_upper = compute_confidence_interval(all_roc_auc_cv)
roc_auc_test_mean, roc_auc_test_lower, roc_auc_test_upper = compute_confidence_interval(all_roc_auc_test)
pr_auc_test_mean, pr_auc_test_lower, pr_auc_test_upper = compute_confidence_interval(all_pr_auc_test)
sensitivity_test_mean, sensitivity_test_lower, sensitivity_test_upper = compute_confidence_interval(all_sensitivity_test)
specificity_test_mean, specificity_test_lower, specificity_test_upper = compute_confidence_interval(all_specificity_test)
f1_test_mean, f1_test_lower, f1_test_upper = compute_confidence_interval(all_f1_test)
ppv_test_mean, ppv_test_lower, ppv_test_upper = compute_confidence_interval(all_ppv_test)
npv_test_mean, npv_test_lower, npv_test_upper = compute_confidence_interval(all_npv_test)
brier_test_mean, brier_test_lower, brier_test_upper = compute_confidence_interval(all_brier_test)

results_data.append({'Кросс-валидационный ROC-AUC [95% ДИ]': format_confidence_interval(roc_auc_cv_mean, roc_auc_cv_lower, roc_auc_cv_upper),
                     'Тестовый ROC-AUC [95% ДИ]': format_confidence_interval(roc_auc_test_mean, roc_auc_test_lower,roc_auc_test_upper),
                     'Тестовый  RAUC [95% ДИ]': format_confidence_interval(pr_auc_test_mean, pr_auc_test_lower,pr_auc_test_upper),
                     'Тестовый Sensitivity [95% ДИ]': format_confidence_interval(sensitivity_test_mean, sensitivity_test_lower,sensitivity_test_upper),
                     'Тестовый Specificity [95% ДИ]': format_confidence_interval(specificity_test_mean, specificity_test_lower,specificity_test_upper),
                     'Тестовый F1-score [95% ДИ]': format_confidence_interval(f1_test_mean, f1_test_lower,f1_test_upper),
                     'Тестовый PPV [95% ДИ]': format_confidence_interval(ppv_test_mean, ppv_test_lower,ppv_test_upper),
                     'Тестовый NPV [95% ДИ]': format_confidence_interval(npv_test_mean, npv_test_lower,npv_test_upper),
                     'Тестовый Brier Score [95% ДИ]': format_confidence_interval(brier_test_mean, brier_test_lower,brier_test_upper),
})

#создание DataFrame после цикла
results_df = pd.DataFrame(results_data)
print(results_df)

Лучшая модель — итерация 59 с ROC-AUC на тесте = 0.8899
  Кросс-валидационный ROC-AUC [95% ДИ] Тестовый ROC-AUC [95% ДИ]  \
0              0.7372 [0.7305; 0.7439]   0.7398 [0.7269; 0.7528]   

   Тестовый  RAUC [95% ДИ] Тестовый Sensitivity [95% ДИ]  \
0  0.1333 [0.1234; 0.1432]       0.5523 [0.5262; 0.5785]   

  Тестовый Specificity [95% ДИ] Тестовый F1-score [95% ДИ]  \
0       0.7935 [0.7862; 0.8008]    0.1435 [0.1363; 0.1506]   

     Тестовый PPV [95% ДИ]    Тестовый NPV [95% ДИ]  \
0  0.0828 [0.0785; 0.0870]  0.9817 [0.9807; 0.9827]   

  Тестовый Brier Score [95% ДИ]  
0       0.0310 [0.0310; 0.0310]  
